In [3]:
"""Step 2a: assign Illumina 450K probe IDs (IlmnID) to the 2,000 CpG columns."""

import pandas as pd
from google.colab import files

N_COHORT_EXPECTED = 2546    # cohort size stated in Methods
N_CPG_EXPECTED = 2000       # CpG columns in the matrix


def add_authentic_probe_ids_and_save(methylation_file, annotation_file):
    """Assign probe IDs from the annotation file to the methylation matrix and save it.

    NOTE: probe IDs are assigned by POSITION. Column j of the matrix is labelled
    with the j-th unique IlmnID of the manifest (the first 2,000 IlmnIDs).
    """

    print("Reading annotation data...")
    annotation_data = pd.read_csv(annotation_file, skiprows=7, low_memory=False)

    # Extract valid probe IDs
    if "IlmnID" not in annotation_data.columns:
        raise ValueError("Annotation file does not contain a 'IlmnID' column.")

    valid_probe_ids = annotation_data["IlmnID"].dropna().unique().tolist()

    print("Reading methylation data...")

    meth_data = pd.read_csv(methylation_file, header=None)
    print(f"Methylation matrix shape: {meth_data.shape}")
    assert meth_data.shape == (N_COHORT_EXPECTED, N_CPG_EXPECTED), (
        f"Expected {(N_COHORT_EXPECTED, N_CPG_EXPECTED)}, found {meth_data.shape}")

    num_probes = len(meth_data.columns)

    if num_probes > len(valid_probe_ids):
        raise ValueError(f"Not enough probe IDs in the annotation file ({len(valid_probe_ids)} available, {num_probes} needed).")

    # Assign probe IDs (by position)
    assigned_probe_ids = valid_probe_ids[:num_probes]

    # Create new dataframe with probe IDs as column names
    new_meth_data = pd.DataFrame(
        meth_data.values,
        columns=assigned_probe_ids,
        index=meth_data.index
    )

    # Save new file (index is kept: PCA.ipynb reads it with index_col=0)
    new_file = 'methylation_with_probeids.csv'
    new_meth_data.to_csv(new_file, index=True)
    print(f"Saved new methylation data with probe IDs to: {new_file}")

    # Download the result
    files.download(new_file)


# Upload methylation data
print("Please upload your methylation CSV file (DNAm_processed_imputed.csv)...")
uploaded_meth = files.upload()
methylation_file = list(uploaded_meth.keys())[0]

# Upload annotation file
print("Please upload the annotation CSV file (humanmethylation450_15017482_v1-2.csv)...")
uploaded_anno = files.upload()
annotation_file = list(uploaded_anno.keys())[0]

# Add probe IDs and save
add_authentic_probe_ids_and_save(methylation_file, annotation_file)

Please upload your methylation CSV file (DNAm_processed_imputed.csv)...


Saving DNAm_processed_imputed.csv to DNAm_processed_imputed.csv
Please upload the annotation CSV file (humanmethylation450_15017482_v1-2.csv)...


Saving humanmethylation450_15017482_v1-2.csv.crdownload to humanmethylation450_15017482_v1-2.csv.crdownload
Reading annotation data...
Reading methylation data...
Methylation matrix shape: (2546, 2000)
Saved new methylation data with probe IDs to: methylation_with_probeids.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
import numpy as np
import pandas as pd
from google.colab import files

def create_mapping(methylation_file, path, anno_data):
    """Create gene to DNAm mapping"""
    # Read methylation data to get probe IDs
    print("Reading methylation data...")
    meth_data = pd.read_csv(methylation_file)
    probe_ids = set(meth_data.columns)  # Convert to set for faster lookup

    if anno_data is None:
        return

    # Create mapping dictionary
    print("Creating mapping...")
    mapping = {}

    for i in range(len(anno_data)):
        if i % 10000 == 0:  # Progress update every 10000 rows
            print(f"{i+1}/{len(anno_data)}")

        probe_id = anno_data.iloc[i]['probeID']
        gene = anno_data.iloc[i]['gene']

        # Skip if probe isn't in our methylation data
        if probe_id not in probe_ids:
            continue

        if pd.isna(gene):  # no mapped gene
            continue
        elif ';' in str(gene):  # multiple mapped genes
            genes = gene.split(';')
            genes = set(genes)  # remove duplicate gene names
            for gene in genes:
                if gene in mapping:
                    mapping[gene].append(probe_id)
                else:
                    mapping[gene] = [probe_id]
        else:  # single mapped gene
            if gene in mapping:
                mapping[gene].append(probe_id)
            else:
                mapping[gene] = [probe_id]

    # Write mapping to file
    output_file = 'gene2DNAm.map'
    print("Writing mapping to file...")
    with open(output_file, 'w') as f:
        for gene in mapping.keys():
            line = gene + '\t' + '\t'.join(mapping[gene])
            f.write(line + '\n')

    print("Mapping complete!")
    files.download(output_file)  # Download the output file
    return mapping

# --- Upload input files ---
print("Please upload your methylation CSV file (methylation_with_probeids.csv)...")
uploaded_meth = files.upload()
methylation_file = list(uploaded_meth.keys())[0]

print("Please upload your annotation CSV file (humanmethylation450_15017482_v1-2.csv)...")
uploaded_anno = files.upload()
annotation_file = list(uploaded_anno.keys())[0]

# --- Process annotation file ---
anno_data = pd.read_csv(annotation_file, skiprows=7)
anno_data = anno_data[['IlmnID', 'UCSC_RefGene_Name']]
anno_data.columns = ['probeID', 'gene']

# --- Create mapping ---
mapping = create_mapping(methylation_file, '', anno_data)

Please upload your methylation CSV file (methylation_with_probeids.csv)...


Saving methylation_with_probeids.csv to methylation_with_probeids (1).csv
Please upload your annotation CSV file (humanmethylation450_15017482_v1-2.csv)...


Saving humanmethylation450_15017482_v1-2.csv.crdownload to humanmethylation450_15017482_v1-2.csv (1).crdownload
Reading methylation data...
Creating mapping...
1/184331
10001/184331
20001/184331
30001/184331
40001/184331
50001/184331
60001/184331
70001/184331
80001/184331
90001/184331
100001/184331
110001/184331
120001/184331
130001/184331
140001/184331
150001/184331
160001/184331
170001/184331
180001/184331
Writing mapping to file...
Mapping complete!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>